In [1]:
import torch
from torch import nn
import os
from os import path
import torchvision
import torchvision.transforms as T
from typing import Sequence
from torchvision.transforms import functional as F
import numbers
import random
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
import torchmetrics as TM
from torch.utils.data import Dataset, random_split
from resource.branch_dataset.dataset import BranchData

# Convert a pytorch tensor into a PIL image
t2img = T.ToPILImage()
# Convert a PIL image into a pytorch tensor
img2t = T.ToTensor()

In [12]:
# Model definition. We use a SegNet-Basic model with some minor tweaks.
# Our input images are 128x128.
class DownConv2(nn.Module):
    def __init__(self, chin, chout, kernel_size):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_channels=chin, out_channels=chout, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chout),
            nn.ReLU(),
            nn.Conv2d(in_channels=chout, out_channels=chout, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chout),
            nn.ReLU(),
        )
        self.mp = nn.MaxPool2d(kernel_size=2, return_indices=True)
        
    def forward(self, x):
        y = self.seq(x)
        pool_shape = y.shape
        y, indices = self.mp(y)
        return y, indices, pool_shape

class DownConv3(nn.Module):
    def __init__(self, chin, chout, kernel_size):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_channels=chin, out_channels=chout, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chout),
            nn.ReLU(),
            nn.Conv2d(in_channels=chout, out_channels=chout, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chout),
            nn.ReLU(),
            nn.Conv2d(in_channels=chout, out_channels=chout, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chout),
            nn.ReLU(),
        )
        self.mp = nn.MaxPool2d(kernel_size=2, return_indices=True)
        
    def forward(self, x):
        y = self.seq(x)
        pool_shape = y.shape
        y, indices = self.mp(y)
        return y, indices, pool_shape

class UpConv2(nn.Module):
    def __init__(self, chin, chout, kernel_size):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_channels=chin, out_channels=chin, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chin),
            nn.ReLU(),
            nn.Conv2d(in_channels=chin, out_channels=chout, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chout),
            nn.ReLU(),
        )
        self.mup = nn.MaxUnpool2d(kernel_size=2)
        
    def forward(self, x, indices, output_size):
        y = self.mup(x, indices, output_size=output_size)
        y = self.seq(y)
        return y

class UpConv3(nn.Module):
    def __init__(self, chin, chout, kernel_size):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_channels=chin, out_channels=chin, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chin),
            nn.ReLU(),
            nn.Conv2d(in_channels=chin, out_channels=chin, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chin),
            nn.ReLU(),
            nn.Conv2d(in_channels=chin, out_channels=chout, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chout),
            nn.ReLU(),
        )
        self.mup = nn.MaxUnpool2d(kernel_size=2)
        
    def forward(self, x, indices, output_size):
        y = self.mup(x, indices, output_size=output_size)
        y = self.seq(y)
        return y

class UpConv2Final(nn.Module):
    def __init__(self, chin, chout, kernel_size):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_channels=chin, out_channels=chin, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chin),
            nn.ReLU(),
            nn.Conv2d(in_channels=chin, out_channels=chout, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(chout),
            nn.Sigmoid(),
        )
        self.mup = nn.MaxUnpool2d(kernel_size=2)
        
    def forward(self, x, indices, output_size):
        y = self.mup(x, indices, output_size=output_size)
        y = self.seq(y)
        return y

class ImageSegmentation(torch.nn.Module):
    def __init__(self, layers, kernel_size):
        super().__init__()
        self.out_channels = 1
        in_channels = 3
        self.bn_input = nn.BatchNorm2d(in_channels)

        self.downconv = []
        for (n_conv, chn_out) in layers:
            if n_conv == 2:
                self.downconv.append(
                    DownConv2(in_channels, chn_out, kernel_size=kernel_size)
                )
            else:
                self.downconv.append(
                    DownConv3(in_channels, chn_out, kernel_size=kernel_size)
                )
            in_channels = chn_out

        self.upconv = []
        for (n_conv, chn_out) in layers[::-1]:
            if n_conv == 2:
                self.upconv.append(
                    UpConv2(in_channels, chn_out, kernel_size=kernel_size)
                )
            else:
                self.upconv.append(
                    UpConv3(in_channels, chn_out, kernel_size=kernel_size)
                )
            in_channels = chn_out
        self.upconv.append(UpConv2Final(in_channels, self.out_channels, kernel_size=kernel_size))
        
        
    def forward(self, batch: torch.Tensor):
        x = self.bn_input(batch)
        # x = batch
        # SegNet Encoder
        mp_indices = []
        shape = []
        for layer in self.downconv:
            (x, _mp_indices, _shape) = layer(x)
            mp_indices.append(_mp_indices)
            shape.append(_shape)

        # SegNet Decoder
        for i in range(len(self.upconv)):
            x = self.upconv[i](x, mp_indices[-1], output_size=shape[-1])
            del mp_indices[-1]
            del shape[-1]
       
        return x
    # end def
# end class

In [10]:
dataset = BranchData(
    os.path.join("resource", "branch_dataset"), 
    transform = T.Compose([
        T.Resize((128, 128), interpolation=T.InterpolationMode.NEAREST)
    ])
)
train_set, test_set = random_split(dataset, [0.9, 0.1])

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

In [13]:
m = ImageSegmentation(layers=[(2, 64), (2, 128), (2, 256)], kernel_size=3)
m.eval()
(batch_in, batch_out) = next(iter(test_loader))
m(batch_in.float()).shape

ValueError: invalid output_size "torch.Size([128, 128])" (dim 0 must be between 30 and 34)

In [ ]:
# Train the model for a single epoch
def train_model(model, loader, optimizer):
    model.train() # Activate "training mode"
    criterion = nn.BCELoss()
    # end if

    running_loss = 0.0
    running_samples = 0
    
    for batch_idx, (inputs, targets) in enumerate(loader, 0):
        optimizer.zero_grad()
        outputs = model(inputs.float())
        
        loss = criterion(outputs.float(), targets.float())
        loss.backward()
        optimizer.step()
    
        running_samples += targets.size(0)
        running_loss += loss.item()
    # end for

    print("Trained {} samples, Loss: {:.4f}".format(
        running_samples,
        running_loss / (batch_idx+1),
    ))

    return running_loss / (batch_idx+1)
# end def

In [ ]:
def test_model(model, loader):
    model.eval()
    accuracies = []
    
    for batch_idx, (inputs, targets) in enumerate(loader, 0):
        predictions = model(inputs.float())
        
        loss = nn.BCELoss(predictions.float(), targets.float())
        accuracies.append(1.0 - loss.item())
        
    return accuracies

In [ ]:
# Optimizer and Learning Rate Scheduler.
optimizer = torch.optim.Adam(m.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.7)

In [ ]:
# Define training loop. This will train the model for multiple epochs.
#
# epochs: A tuple containing the start epoch (inclusive) and end epoch (exclusive).
#         The model is trained for [epoch[0] .. epoch[1]) epochs.
#
def train_loop(model, train_loader, test_loader, epochs, optimizer, scheduler):
    epoch_i, epoch_j = epochs
    loss = []
    for i in range(epoch_i, epoch_j):
        epoch = i
        print(f"Epoch: {i:02d}, Learning Rate: {optimizer.param_groups[0]['lr']}")
        train_loss = train_model(model, train_loader, optimizer)
        # end with

        if scheduler is not None:
            scheduler.step()
        # end if

        test_acc = np.mean(test_model(model, test_loader))
        loss.append((train_loss, test_acc))
        print(f"Test set accurracy: {test_acc}")

        print("")

    return loss
    # end for
# end def